# 02 · Kokoro TTS: Open-Source Speech Synthesis at 82M Parameters

**Hardware**: 🟢 CPU works (the model is only 82M params / ~330MB; real-time synthesis on a laptop)

## What you will learn

1. Run Kokoro-82M: top of TTS Arena, Apache 2.0, real-time on CPU — the value king of open TTS
2. Its technical route: **non-autoregressive** (StyleTTS2 style) vs. this chapter's codec-LM paradigm, and the trade-offs
3. Languages, voices, speed control
4. Round-trip testing with Whisper (TTS → ASR) to verify synthesis quality automatically

## Prerequisites

Some languages need espeak-ng for phonemization:
- macOS: `brew install espeak-ng`
- Ubuntu: `sudo apt-get install espeak-ng`
- Chinese support: `pip install "misaki[zh]"`

In [ ]:
%pip install -q "kokoro>=0.9.4" soundfile
# For Chinese, also: %pip install -q "misaki[zh]"

## 1. English synthesis

`KPipeline`'s `lang_code` selects the phonemization frontend: `'a'`=American English, `'b'`=British, `'z'`=Chinese, `'j'`=Japanese.

Kokoro is **non-autoregressive**: text → phonemes → duration prediction → the whole waveform in one shot. Upside: fast and stable (no autoregressive stutters or dropped words). Downside: none of the codec-LM route's zero-shot cloning or contextual expressiveness — hold this trade-off against [theory.md](../theory.md) §3.

In [ ]:
from kokoro import KPipeline
from IPython.display import Audio as AudioPlayer, display
import soundfile as sf
import numpy as np

pipe_en = KPipeline(lang_code="a")  # American English

text_en = (
    "Kokoro is an open source text to speech model with only eighty two million parameters. "
    "It runs in real time on a laptop CPU, and it is licensed under Apache two point zero."
)

chunks = []
for gs, ps, audio in pipe_en(text_en, voice="af_heart"):
    print(f"text chunk: {gs[:60]}..." if len(gs) > 60 else f"text chunk: {gs}")
    print(f"phonemes: {ps[:80]}")
    chunks.append(audio)

wav_en = np.concatenate(chunks)
sf.write("kokoro_en.wav", wav_en, 24000)
display(AudioPlayer(wav_en, rate=24000))

## 2. Voices and speed

Kokoro ships 54 voices (voice embeddings, not separate models). Naming: `af_*` American female, `am_*` American male, `bf_*/bm_*` British, `zf_*/zm_*` Chinese.

In [ ]:
demo = "The same sentence, three different voices."

for voice in ["af_heart", "af_bella", "am_michael"]:
    audio = np.concatenate([a for _, _, a in pipe_en(demo, voice=voice)])
    print(f"voice = {voice}")
    display(AudioPlayer(audio, rate=24000))

# Speed control
for speed in [0.8, 1.0, 1.4]:
    audio = np.concatenate([a for _, _, a in pipe_en(demo, voice="af_heart", speed=speed)])
    print(f"speed = {speed}")
    display(AudioPlayer(audio, rate=24000))

## 3. Chinese synthesis

Requires `pip install "misaki[zh]"` (the Chinese G2P frontend). The hard part of Chinese TTS is polyphonic characters — G2P frontend quality sets the ceiling.

In [ ]:
try:
    pipe_zh = KPipeline(lang_code="z")
    text_zh = "多模态一零一是一个兼具理论与实践的开源教程，重点是把原理学清楚。"
    wav_zh = np.concatenate([a for _, _, a in pipe_zh(text_zh, voice="zf_xiaobei")])
    sf.write("kokoro_zh.wav", wav_zh, 24000)
    display(AudioPlayer(wav_zh, rate=24000))
except Exception as e:
    print(f"Chinese pipeline init failed (most likely missing misaki[zh]): {e}")

## 4. Round-trip test: TTS → ASR automatic verification

How do you batch-verify TTS quality without ears? Feed the synthesized audio back into Whisper and diff the transcript against the source text — **high WER means pronunciation problems**. A common regression-testing trick in production (it can't judge timbre/naturalness, though — that still needs human or Audio-Turing-Test style evaluation).

In [ ]:
%pip install -q transformers jiwer
import jiwer
from transformers import pipeline as hf_pipeline

asr = hf_pipeline("automatic-speech-recognition", model="openai/whisper-small")
hyp = asr("kokoro_en.wav")["text"]

norm = jiwer.Compose([jiwer.ToLowerCase(), jiwer.RemovePunctuation(),
                      jiwer.RemoveMultipleSpaces(), jiwer.Strip()])
wer = jiwer.wer(norm(text_en), norm(hyp))
print(f"source:   {text_en}")
print(f"ASR read: {hyp}")
print(f"round-trip WER: {wer:.2%}")

## Exercises

1. Numbers, abbreviations and URLs are classic TTS traps: have Kokoro read `"GPT-5 costs $1.25 per 1M tokens, see https://example.com"`, hear where it breaks, and think about what text normalization must do.
2. Script a converter that reads the whole of [theory.md](../theory.md) as a podcast (mind long-text chunking).
3. Comparative study: synthesize the same passage with Kokoro and Higgs Audio v3 (`03_higgs_v3_clone.ipynb`, planned) and score naturalness/expressiveness/speed — feel the non-autoregressive vs. codec-LM divide.

**Next stop**: `04_voice_pipeline.ipynb` (planned) — chain Whisper + LLM + Kokoro into full voice chat.